# Pipeline MLOps con MLflow: Predicción y Clasificación de Calidad de Vinos

**Asignatura:** Ciencia de Datos  
**Institución:** Universidad de Cuenca  
**Integrantes:** Joseph Salamea & Xavier Peña  
**Dataset:** Wine Quality (UCI Machine Learning Repository, ID 186, Cortez et al., 2009)  
**Herramienta Principal:** MLflow (Open Source, v2.x)  
**Herramienta Comparativa:** Weights & Biases (W&B)  

---

## Estructura del Pipeline
1. **Configuración del Entorno y Librerías**
2. **Carga y Estructuración de Datos (Vino Tinto y Blanco)**
3. **Control de Calidad: Valores Nulos, Tipos y Consistencia**
4. **Análisis Estadístico Descriptivo**
5. **Distribución de Calidad y Formulación del Problema de Clasificación**
6. **Análisis de Correlación Fisicoquímica**
7. **Perfiles Fisicoquímicos: Vinos Tintos vs. Vinos Blancos**
8. **Detección de Patrones y Outliers**
9. **Preprocesamiento y Partición Estratificada (Train / Test)**
10. **Síntesis de Hallazgos y Decisiones para el Modelado MLOps**


## 1. Configuración del Entorno y Librerías

In [ ]:
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Configuración de estilos visuales y advertencias
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Fijación de semilla aleatoria para garantizar reproducibilidad determinista
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas y entorno configurado con reproducibilidad fijada.")


## 2. Carga y Estructuración de Datos

El conjunto de datos original de Cortez et al. (2009) contiene muestras correspondientes a variantes de vino verde portugués (*Vinho Verde*):
* **Vino Tinto (`winequality-red.csv`):** 1.599 instancias.
* **Vino Blanco (`winequality-white.csv`):** 4.898 instancias.

Ambas fuentes comparten 11 variables fisicoquímicas continuas y una variable sensorial objetivo (`quality`, puntaje de 0 a 10). Siguiendo las mejores prácticas, incorporamos un atributo indicativo `wine_type` y concatenamos ambas variantes en un dataset consolidado de **6.497 instancias**.


In [ ]:
# Definir rutas relativas con soporte para ejecución desde /notebooks o desde la raíz
path_red = os.path.join('..', 'data', 'winequality-red.csv')
path_white = os.path.join('..', 'data', 'winequality-white.csv')

if not os.path.exists(path_red):
    path_red = os.path.join('data', 'winequality-red.csv')
    path_white = os.path.join('data', 'winequality-white.csv')

# Cargar datasets con separador ';'
df_red = pd.read_csv(path_red, sep=';')
df_white = pd.read_csv(path_white, sep=';')

# Agregar variable categórica de tipo de vino
df_red['wine_type'] = 'red'
df_white['wine_type'] = 'white'

# Concatenar en DataFrame unificado
df = pd.concat([df_red, df_white], ignore_index=True)

print("Carga de datos completada:")
print(f" - Vinos Tintos:  {df_red.shape[0]} filas, {df_red.shape[1]} columnas")
print(f" - Vinos Blancos: {df_white.shape[0]} filas, {df_white.shape[1]} columnas")
print(f" - Dataset Total: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()


## 3. Control de Calidad: Valores Nulos, Tipos y Consistencia

Verificamos la integridad de los datos para garantizar que no existan valores nulos, registros corruptos o tipos incompatibles que alteren las fases posteriores del pipeline.


In [ ]:
# Inspección de tipos de datos, nulos y cardinalidad
data_audit = pd.DataFrame({
    'Tipo de Dato': df.dtypes,
    'Valores Nulos': df.isnull().sum(),
    '% Nulos': (df.isnull().sum() / len(df)) * 100,
    'Valores Únicos': df.nunique()
})

print("Auditoría de Integridad del Dataset:")
print(data_audit)

# Evaluación de filas duplicadas en variables fisicoquímicas
num_duplicates = df.duplicated().sum()
print(f"\nTotal de registros con características fisicoquímicas idénticas: {num_duplicates} ({(num_duplicates / len(df))*100:.2f}%)")
print("Nota: En enología, es factible que diferentes catas o lotes compartan métricas de laboratorio idénticas, por lo que se preservan para respetar la distribución empírica.")


## 4. Análisis Estadístico Descriptivo

Analizamos los estadísticos descriptivos (media, desviación estándar, rango intercuartílico, mínimos y máximos) para entender la dispersión y escala de cada variable fisicoquímica.


In [ ]:
# Resumen estadístico numérico redondeado
numeric_cols = df.select_dtypes(include=[np.number]).columns
stats_summary = df[numeric_cols].describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
stats_summary.round(3)


## 5. Distribución de Calidad y Formulación del Problema de Clasificación

La variable sensorial `quality` varía de 3 a 9 en los datos observados (no hay registros con 0, 1, 2 ni 10).
Conforme a la literatura estándar sobre el dataset de Cortez et al. (2009):
* La gran mayoría de los vinos se concentran en las categorías medias (5 y 6).
* Formulamos un problema de **clasificación binaria**:
  * **Clase 1 (Buena Calidad):** `quality >= 6` (vinos calificados como buenos o excepcionales).
  * **Clase 0 (Calidad Regular / Baja):** `quality < 6` (vinos calificados con puntajes 3, 4 o 5).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Distribución de puntuaciones discretas
sns.countplot(data=df, x='quality', hue='wine_type', palette=['#722F37', '#E6C687'], ax=axes[0])
axes[0].set_title('Distribución de Puntuaciones Sensoriales (3 a 9) por Tipo', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Puntuación de Calidad (quality)')
axes[0].set_ylabel('Número de Muestras')
axes[0].legend(title='Tipo de Vino', labels=['Tinto', 'Blanco'])

# 2. Creación y distribución de la variable binaria
df['good_quality'] = (df['quality'] >= 6).astype(int)

sns.countplot(data=df, x='good_quality', hue='wine_type', palette=['#722F37', '#E6C687'], ax=axes[1])
axes[1].set_title('Balance de Clases: Regular (<6) vs. Buena Calidad (>=6)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Target Binario (good_quality)')
axes[1].set_ylabel('Número de Muestras')
axes[1].set_xticklabels(['Regular / Baja (<6)', 'Buena Calidad (>=6)'])
axes[1].legend(title='Tipo de Vino', labels=['Tinto', 'Blanco'])

plt.tight_layout()
plt.show()

# Conteo y proporciones exactas
counts = df['good_quality'].value_counts()
props = df['good_quality'].value_counts(normalize=True) * 100
print("Balance del Target Binario (good_quality):")
print(f" - Clase 1 (Buena Calidad >= 6): {counts[1]} vinos ({props[1]:.2f}%)")
print(f" - Clase 0 (Regular / Baja < 6):  {counts[0]} vinos ({props[0]:.2f}%)")


## 6. Análisis de Correlación Fisicoquímica

Evaluamos la matriz de correlación lineal de Pearson entre todas las variables predictoras y la calidad. Esto permite identificar:
1. Qué atributos fisicoquímicos tienen mayor correlación directa o inversa con la calidad del vino.
2. Si existen variables fuertemente colineales entre sí (ej. densidad vs. alcohol, dióxido de azufre libre vs. total).


In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()

# Máscara para la parte superior del triángulo
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(12, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='vlag', vmin=-0.8, vmax=0.8,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlación Fisicoquímica de Pearson (Vinho Verde)', fontsize=14, pad=12, fontweight='bold')
plt.show()

# Ranking de correlación contra la calidad
quality_corr = corr['quality'].drop(['quality', 'good_quality']).sort_values(ascending=False)
print("Correlación de atributos fisicoquímicos con 'quality':")
print(quality_corr.round(4))


## 7. Perfiles Fisicoquímicos: Vinos Tintos vs. Vinos Blancos

El vino tinto y el vino blanco difieren en sus procesos de fermentación y maceración con la piel de la uva. Comparamos las cuatro variables químicas con mayor divergencia:
* `alcohol`
* `volatile acidity` (acidez volátil)
* `residual sugar` (azúcar residual)
* `total sulfur dioxide` (dióxido de azufre total)


In [ ]:
features_compare = ['alcohol', 'volatile acidity', 'residual sugar', 'total sulfur dioxide']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(features_compare):
    sns.boxplot(data=df, x='wine_type', y=col, palette=['#722F37', '#E6C687'], ax=axes[i], width=0.4)
    axes[i].set_title(f'Distribución de {col.title()}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Tipo de Vino')
    axes[i].set_xticklabels(['Tinto', 'Blanco'])
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

# Comparación numérica de medias
mean_compare = df.groupby('wine_type')[features_compare].mean().T
mean_compare.columns = ['Vino Tinto (Media)', 'Vino Blanco (Media)']
mean_compare['Diferencia Absoluta'] = (mean_compare['Vino Blanco (Media)'] - mean_compare['Vino Tinto (Media)']).abs()
print("Comparativa de Medias Fisicoquímicas por Tipo:")
print(mean_compare.round(3))


## 8. Detección de Patrones y Relación con la Calidad

Visualizamos cómo interactúan los dos principales factores correlacionados (`alcohol` y `volatile acidity`) con la calidad binaria del vino mediante gráficos de distribución y violines.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Gráfico de violín: Alcohol vs Calidad
sns.violinplot(data=df, x='good_quality', y='alcohol', hue='wine_type', split=True,
               palette=['#722F37', '#E6C687'], ax=axes[0])
axes[0].set_title('Alcohol por Calidad y Tipo de Vino', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Calidad del Vino')
axes[0].set_xticklabels(['Regular (<6)', 'Buena (>=6)'])
axes[0].set_ylabel('Alcohol (% vol)')
axes[0].legend(title='Tipo', labels=['Tinto', 'Blanco'])

# Gráfico de violín: Acidez Volátil vs Calidad
sns.violinplot(data=df, x='good_quality', y='volatile acidity', hue='wine_type', split=True,
               palette=['#722F37', '#E6C687'], ax=axes[1])
axes[1].set_title('Acidez Volátil por Calidad y Tipo de Vino', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Calidad del Vino')
axes[1].set_xticklabels(['Regular (<6)', 'Buena (>=6)'])
axes[1].set_ylabel('Acidez Volátil (g/dm³)')
axes[1].legend(title='Tipo', labels=['Tinto', 'Blanco'])

plt.tight_layout()
plt.show()


## 9. Preprocesamiento y Partición Estratificada (Train / Test)

Para alimentar los experimentos de MLflow y evitar fuga de información (*data leakage*):
1. **Codificación:** Transformamos `wine_type` en una variable binaria numérica (`is_white_wine = 1` para blanco, `0` para tinto).
2. **Definición de Matrices:** Separamos la matriz de variables predictoras $X$ (12 variables) y el vector de etiquetas $y$ (`good_quality`).
3. **Partición Estratificada:** Dividimos el dataset en **80% Entrenamiento (Train)** y **20% Prueba (Test)** garantizando idéntica proporción de clases positivas mediante `stratify=y`.


In [ ]:
# 1. Codificación de variables
df_processed = df.copy()
df_processed['is_white_wine'] = (df_processed['wine_type'] == 'white').astype(int)

# 2. Selección de columnas de características predictoras
feature_cols = [
    'fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
    'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
    'pH', 'sulphates', 'alcohol', 'is_white_wine'
]

X = df_processed[feature_cols]
y = df_processed['good_quality']

# 3. Partición estratificada Train (80%) y Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Dimensiones de los conjuntos preparados para Modelado MLOps:")
print(f" - X_train: {X_train.shape} (80%) | y_train: {y_train.shape}")
print(f" - X_test:  {X_test.shape} (20%) | y_test:  {y_test.shape}")
print("\nVerificación de distribución de clases (Estratificación exitosa):")
print(f" - Train ratio vinos buenos (clase 1): {y_train.mean():.4f} ({y_train.sum()} muestras)")
print(f" - Test ratio vinos buenos (clase 1):  {y_test.mean():.4f} ({y_test.sum()} muestras)")


## 10. Síntesis de Hallazgos y Decisiones para el Modelado MLOps

A partir del análisis exploratorio de datos, fundamentamos las siguientes decisiones técnicas para la suite de experimentos:

| Hallazgo en el EDA | Impacto / Decisión para el Modelado |
| :--- | :--- |
| **Alcohol (+0.44 correlación):** Factor más determinante de alta calidad sensorial. | Será una variable con alta importancia de características (*feature importance*) en árboles y coeficientes positivos en modelos lineales. |
| **Acidez Volátil (-0.27 correlación):** Asociada a defectos del vino (olor a vinagre/ácido acético). | Crucial para penalizar vinos de baja calidad. |
| **Diferencias Marcadas Tinto vs. Blanco:** Vinos blancos tienen 3x más dióxido de azufre y azúcar residual; tintos tienen mayor acidez volátil. | Incluir `is_white_wine` como predictor permite al modelo ajustar los umbrales químicos según la variante del vino. |
| **Escalas Heterogéneas:** Variables en rangos de 0.01 a 280 (ej. cloruros vs dióxido de azufre total). | **Obligatorio escalar (`StandardScaler`)** para modelos sensibles a la escala como Regresión Logística y Ridge. Los modelos de ensamble basados en árboles (Random Forest y XGBoost) son invariantes a la escala monótona. |
| **Balance Moderado de Clases:** 63.3% clase positiva (>=6) vs. 36.7% clase regular (<6). | Emplearemos **F1-Score (Macro)**, **ROC-AUC** y **Accuracy** como métricas principales registradas en MLflow para evaluar el trade-off precisión/recall. |

---

## 11. Próxima Etapa: Seguimiento de Experimentos con MLflow

El conjunto de datos procesado `(X_train, X_test, y_train, y_test)` queda listo para ejecutar la configuración de MLflow:
1. Inicialización del servidor de tracking y creación del experimento `wine-quality-mlops`.
2. Ejecución y registro estructurado de **10+ experimentos**:
   * Experimentos 1-3: Regresión Logística y Ridge con diferentes penalizaciones (`C`, `alpha`) y escalado estándar.
   * Experimentos 4-7: Random Forest variando `n_estimators`, `max_depth` y `min_samples_split`.
   * Experimentos 8-10+: XGBoost Classifier con búsqueda de tasa de aprendizaje (`learning_rate`), `max_depth` y regularización.
3. Comparación de curvas ROC, matrices de confusión y métricas en la UI de MLflow (`mlflow ui`).
4. Selección y promoción formal del mejor modelo al **MLflow Model Registry** en estado `Production`.
5. Ejecución del benchmark comparativo con **Weights & Biases (W&B)** midiendo sobrecarga de latencia y autonomía offline.


## 12. Configuración de MLflow Tracking y Entorno de Reproducibilidad (Xavier Peña)

Para garantizar la gobernanza del ciclo de vida del modelo y habilitar el **MLflow Model Registry**, configuramos el backend de almacenamiento con SQLite (`sqlite:///mlflow.db`). Un backend de base de datos relacional es un requisito fundamental de MLflow para gestionar versiones, etapas y transiciones de estado (`Staging`, `Production`, `Archived`).

En esta etapa:
* Establecemos la URI de seguimiento apuntando a la base de datos SQLite local del proyecto.
* Creamos/seleccionamos el experimento central: `wine-quality-mlops`.
* Configuramos metadatos globales del experimento (asignatura, dataset, autores y objetivo de modelado).

In [ ]:
import os
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# Silenciar mensajes de ayuda de tracing de MLflow
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

# Resolución robusta del directorio raíz del proyecto
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
TRACKING_DB_PATH = os.path.join(BASE_DIR, 'mlflow.db').replace('\\', '/')
TRACKING_URI = f"sqlite:///{TRACKING_DB_PATH}"

# Configuración del servidor de tracking
mlflow.set_tracking_uri(TRACKING_URI)
EXPERIMENT_NAME = "wine-quality-mlops"
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento Activo:  {experiment.name} (ID: {experiment.experiment_id})")
print(f"Almacén de Artefactos: {experiment.artifact_location}")


## 13. Función Modular de Registro Estandarizado (`log_wine_experiment`)

Para evitar duplicidad de código y asegurar que cada corrida registre información exhaustiva, implementamos una función de utilidad que encapsula:
1. **Parámetros del modelo:** Hiperparámetros, tipo de preprocesamiento y semilla aleatoria.
2. **Métricas de rendimiento:** `Accuracy`, `F1-Score (Macro)`, `Precision (Macro)`, `Recall (Macro)`, `ROC-AUC` y `Log-Loss` (cuando el modelo provee probabilidades).
3. **Métricas de entrenamiento:** Para monitorear la brecha de generalización (`overfit_gap_accuracy`) y detectar sobreajuste temprano.
4. **Artefactos visuales autogenerados:**
   * Matriz de confusión anotada con mapa de calor (`confusion_matrix.png`).
   * Curva ROC con su valor AUC correspondiente (`roc_curve.png`).
5. **Empaquetado del modelo:** Serialización con `mlflow.sklearn.log_model` incluyendo la firma de tipos (`signature`), un `input_example` representativo y dependencias explícitas para máxima reproducibilidad.

In [ ]:
import tempfile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, roc_curve
)
from sklearn.pipeline import Pipeline

def log_wine_experiment(
    run_name: str,
    model,
    params: dict,
    tags: dict = None,
    description: str = ""
):
    """
    Entrena, evalúa y registra una corrida completa en MLflow Tracking.
    
    Parámetros:
    -----------
    run_name: str
        Nombre identificador del run.
    model: Estimador o Pipeline de scikit-learn.
    params: dict
        Hiperparámetros a registrar explícitamente.
    tags: dict, opcional
        Metadatos adicionales (ej. autor, categoría, arquitectura).
    description: str, opcional
        Nota descriptiva del experimento.
    """
    default_tags = {
        "author": "Xavier Peña",
        "dataset": "wine-quality-uci",
        "task": "binary_classification",
        "target": "good_quality_ge_6"
    }
    if tags:
        default_tags.update(tags)

    with mlflow.start_run(run_name=run_name, description=description) as run:
        # 1. Registrar hiperparámetros y etiquetas
        mlflow.log_params(params)
        mlflow.set_tags(default_tags)
        
        # 2. Ajuste del modelo con los datos de entrenamiento
        model.fit(X_train, y_train)
        
        # 3. Predicciones en Train y Test
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)
        
        # Obtener probabilidades si están disponibles
        y_test_proba = None
        if hasattr(model, "predict_proba"):
            y_test_proba = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, "decision_function"):
            decision = model.decision_function(X_test)
            y_test_proba = 1 / (1 + np.exp(-decision))
            
        # 4. Cálculo de métricas cuantitativas
        test_acc = accuracy_score(y_test, y_test_pred)
        test_f1_macro = f1_score(y_test, y_test_pred, average='macro')
        test_prec_macro = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
        test_rec_macro = recall_score(y_test, y_test_pred, average='macro')
        train_acc = accuracy_score(y_train, y_train_pred)
        train_f1_macro = f1_score(y_train, y_train_pred, average='macro')
        
        metrics = {
            "test_accuracy": test_acc,
            "test_f1_macro": test_f1_macro,
            "test_precision_macro": test_prec_macro,
            "test_recall_macro": test_rec_macro,
            "train_accuracy": train_acc,
            "train_f1_macro": train_f1_macro,
            "overfit_gap_accuracy": train_acc - test_acc
        }
        
        if y_test_proba is not None:
            metrics["test_roc_auc"] = roc_auc_score(y_test, y_test_proba)
            try:
                metrics["test_log_loss"] = log_loss(y_test, y_test_proba)
            except Exception:
                pass
                
        mlflow.log_metrics(metrics)
        
        # 5. Generar y registrar artefactos visuales
        with tempfile.TemporaryDirectory() as tmp_dir:
            # Artefacto 1: Matriz de Confusión
            cm = confusion_matrix(y_test, y_test_pred)
            fig_cm, ax_cm = plt.subplots(figsize=(6, 5))
            sns.heatmap(
                cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax_cm,
                xticklabels=['Regular (<6)', 'Bueno (>=6)'],
                yticklabels=['Regular (<6)', 'Bueno (>=6)']
            )
            ax_cm.set_title(f'Matriz de Confusión - {run_name}', fontsize=12, fontweight='bold')
            ax_cm.set_xlabel('Predicción')
            ax_cm.set_ylabel('Valor Real')
            plt.tight_layout()
            cm_path = os.path.join(tmp_dir, "confusion_matrix.png")
            fig_cm.savefig(cm_path, dpi=120)
            plt.close(fig_cm)
            mlflow.log_artifact(cm_path, artifact_path="plots")
            
            # Artefacto 2: Curva ROC
            if y_test_proba is not None:
                fpr, tpr, _ = roc_curve(y_test, y_test_proba)
                fig_roc, ax_roc = plt.subplots(figsize=(6, 5))
                ax_roc.plot(fpr, tpr, color='#1f77b4', lw=2, label=f"AUC = {metrics['test_roc_auc']:.4f}")
                ax_roc.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
                ax_roc.set_title(f'Curva ROC - {run_name}', fontsize=12, fontweight='bold')
                ax_roc.set_xlabel('Tasa de Falsos Positivos (FPR)')
                ax_roc.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
                ax_roc.legend(loc='lower right')
                plt.tight_layout()
                roc_path = os.path.join(tmp_dir, "roc_curve.png")
                fig_roc.savefig(roc_path, dpi=120)
                plt.close(fig_roc)
                mlflow.log_artifact(roc_path, artifact_path="plots")
                
        # 6. Serialización del modelo con firma y ejemplo de entrada
        signature = infer_signature(X_test, y_test_pred)
        input_example = X_test.head(3)
        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE,
            pip_requirements=["scikit-learn", "numpy", "pandas"]
        )
        
        print(f"[OK] [{run_name}] registrado exitosamente en MLflow.")
        print(f"   -> Run ID: {run.info.run_id}")
        print(f"   -> Test Accuracy: {test_acc:.4f} | Test F1 Macro: {test_f1_macro:.4f}" + (f" | ROC-AUC: {metrics['test_roc_auc']:.4f}" if 'test_roc_auc' in metrics else ""))
        
        return run.info.run_id, metrics


## 14. Experimentos Base: Versión Inicial de Cada Familia de Modelos (Runs 1 a 3)

Ejecutamos y registramos en MLflow Tracking la **versión base** de cada uno de los tres algoritmos previstos en el alcance del proyecto:

1. **Modelo Base 1 - Regresión Logística (`baseline_logistic_regression`):**
   * Preprocesamiento: Escalado estándar con `StandardScaler`.
   * Hiperparámetros por defecto: Penalización $L_2$, $C=1.0$, `solver='lbfgs'`.
   * Propósito: Establecer la cota inferior lineal con probabilidades calibradas.

2. **Modelo Base 2 - Clasificador Ridge (`baseline_ridge_classifier`):**
   * Preprocesamiento: Escalado estándar con `StandardScaler`.
   * Hiperparámetros por defecto: $lpha=1.0$, `solver='auto'`.
   * Propósito: Evaluar la regularización $L_2$ estricta con minimización analítica de mínimos cuadrados.

3. **Modelo Base 3 - Random Forest (`baseline_random_forest`):**
   * Preprocesamiento: Sin escalado (los árboles de decisión son invariantes a transformaciones monótonas).
   * Hiperparámetros por defecto: `n_estimators=100`, `criterion='gini'`, `max_depth=None`.
   * Propósito: Capturar relaciones no lineales e interacciones entre acidez, alcohol y tipo de vino.

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier

# -------------------------------------------------------------
# 1. Modelo Base 1: Regresión Logística con StandardScaler
# -------------------------------------------------------------
lr_base = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

lr_params = {
    "model_family": "LogisticRegression",
    "scaled": True,
    "penalty": "l2",
    "C": 1.0,
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": RANDOM_STATE
}

run_id_lr, metrics_lr = log_wine_experiment(
    run_name="baseline_logistic_regression",
    model=lr_base,
    params=lr_params,
    tags={"model_type": "linear", "stage": "baseline"},
    description="Regresión Logística base con StandardScaler y regularización L2 por defecto."
)

# -------------------------------------------------------------
# 2. Modelo Base 2: Ridge Classifier con StandardScaler
# -------------------------------------------------------------
ridge_base = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RidgeClassifier(
        alpha=1.0,
        solver='auto',
        random_state=RANDOM_STATE
    ))
])

ridge_params = {
    "model_family": "RidgeClassifier",
    "scaled": True,
    "alpha": 1.0,
    "solver": "auto",
    "random_state": RANDOM_STATE
}

run_id_ridge, metrics_ridge = log_wine_experiment(
    run_name="baseline_ridge_classifier",
    model=ridge_base,
    params=ridge_params,
    tags={"model_type": "linear", "stage": "baseline"},
    description="Ridge Classifier base con StandardScaler y alpha=1.0 por defecto."
)

# -------------------------------------------------------------
# 3. Modelo Base 3: Random Forest Base (Sin escalado)
# -------------------------------------------------------------
rf_base = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_params = {
    "model_family": "RandomForestClassifier",
    "scaled": False,
    "n_estimators": 100,
    "criterion": "gini",
    "max_depth": "None",
    "random_state": RANDOM_STATE
}

run_id_rf, metrics_rf = log_wine_experiment(
    run_name="baseline_random_forest",
    model=rf_base,
    params=rf_params,
    tags={"model_type": "ensemble_tree", "stage": "baseline"},
    description="Random Forest base con 100 estimadores y profundidad libre sin escalado."
)


## 15. Comparativa Inicial de Corridas en MLflow Tracking

Consultamos programáticamente los experimentos registrados en MLflow mediante `mlflow.search_runs()` para analizar y contrastar el desempeño de los modelos base.

In [ ]:
# Consulta programática de las corridas registradas en el experimento
runs_df = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.test_f1_macro DESC"]
)

# Filtrar y ordenar columnas clave para visualización
display_cols = [
    'tags.mlflow.runName',
    'params.model_family',
    'metrics.test_accuracy',
    'metrics.test_f1_macro',
    'metrics.test_roc_auc',
    'metrics.overfit_gap_accuracy',
    'run_id'
]
display_cols = [c for c in display_cols if c in runs_df.columns]

summary_table = runs_df[display_cols].copy()
summary_table.columns = [
    'Run Name', 'Familia', 'Test Acc', 'Test F1 Macro', 'ROC-AUC', 'Overfit Gap', 'Run ID'
]

print("=== TABLA COMPARATIVA DE MODELOS BASE (MLflow Tracking) ===")
display(summary_table.round(4))


### 16. Análisis de Resultados Base y Transición al Commit 5

#### Hallazgos Clave de los Modelos Base:
1. **Modelos Lineales (`LogisticRegression` y `RidgeClassifier`):**
   * Desempeño similar (**Accuracy $\approx 73.9\%$**, **F1-Macro $\approx 0.707$**, **ROC-AUC $\approx 0.806$**).
   * **Brecha de sobreajuste casi nula** (`overfit_gap < 0.01`), lo que confirma que generalizan de forma muy estable pero están limitados por su sesgo lineal.
2. **Modelo de Ensamble (`RandomForestClassifier` Base):**
   * Superioridad notable en capacidad discriminativa (**Accuracy $\approx 84.2\%$**, **F1-Macro $\approx 0.826$**, **ROC-AUC $\approx 0.906$**).
   * **Sobreajuste evidente:** El modelo memoriza el conjunto de entrenamiento (Train Accuracy = 100%, con una brecha de $15.8\%$ respecto al test).

#### Próximos Pasos (Commit 5 - Xavier Peña):
* **Búsqueda de Hiperparámetros Avanzada:**
  * Optimización de **XGBoost Classifier** explorando combinaciones de tasa de aprendizaje (`learning_rate`), profundidad (`max_depth`), `n_estimators` y regularización (`gamma`, `subsample`).
  * Regularización de **Random Forest** (ajuste de `max_depth`, `min_samples_split`, `min_samples_leaf`) para reducir el sobreajuste preservando el alto ROC-AUC.
  * Alcanzar el objetivo de **10+ experimentos formalmente registrados** en MLflow Tracking.
* **MLflow Model Registry:**
  * Identificación del modelo campeón según la métrica compuesta (F1-Macro + ROC-AUC con bajo overfit gap).
  * Registro del modelo en el catálogo centralizado (`models:/wine-quality-champion`), asignación de versión formal y transición a la etapa/alias **`Production`**.